# 08 · Unsupervised Appendix — Isolation Forest

What if we had **no labels**? Isolation Forest flags statistical outliers without seeing `Class`. We compare it to the supervised champion at the **same alert budget** — the only fair comparison, since both cost the analyst team the same number of reviews.

In [1]:
# project-root bootstrap
import os
from pathlib import Path
_p=Path.cwd()
while not (_p/'config'/'config.yaml').exists() and _p!=_p.parent:
    _p=_p.parent
os.chdir(_p); print('working dir:', Path.cwd())

working dir: /Users/asfalanoi/app_2026/fraud_detection


In [2]:
import numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest
from fraud.io import load_pickle, read_parquet

ch  = load_pickle('models/champion_xgb.pkl')
pre, model, t_star, FEATURES = ch['preprocessor'], ch['model'], ch['threshold'], ch['features']
train = read_parquet('data/processed/train.parquet')
test  = read_parquet('data/processed/test.parquet')
X_train = pre.transform(train[FEATURES])
X_test  = pre.transform(test[FEATURES])
y_test  = test.Class.values

In [3]:
# Supervised champion: how many alerts does it raise on test?
proba = model.predict_proba(X_test)[:, 1]
sup_pred = (proba >= t_star).astype(int)
budget = int(sup_pred.sum())
sup_caught = int(((sup_pred == 1) & (y_test == 1)).sum())
total_fraud = int(y_test.sum())
print(f'alert budget (champion) = {budget}  | total frauds in test = {total_fraud}')

alert budget (champion) = 60  | total frauds in test = 49


In [4]:
# Isolation Forest — unsupervised, fit WITHOUT labels
iso = IsolationForest(n_estimators=200, contamination='auto', random_state=0, n_jobs=-1)
iso.fit(X_train)
anom = -iso.score_samples(X_test)          # higher = more anomalous
# Flag the SAME number of transactions as the champion (matched budget)
thr = np.sort(anom)[-budget]
iso_pred = (anom >= thr).astype(int)
iso_caught = int(((iso_pred == 1) & (y_test == 1)).sum())

In [5]:
comp = pd.DataFrame([
    {'method': 'Isolation Forest (unsupervised)', 'alerts': int(iso_pred.sum()),
     'frauds caught': iso_caught, 'recall': round(iso_caught/total_fraud, 3)},
    {'method': 'XGBoost champion (supervised)',   'alerts': budget,
     'frauds caught': sup_caught, 'recall': round(sup_caught/total_fraud, 3)},
])
comp

,method,alerts,frauds caught,recall
0,Isolation Forest (unsupervised),60,16,0.327
1,XGBoost champion (supervised),60,44,0.898


**Conclusion:** at an identical analyst workload, the supervised champion catches far more fraud. Isolation Forest is a useful cold-start when labels are unavailable, but once you have labelled outcomes, supervised learning dominates. It is an appendix, not the system.